# Import libraries

In [ ]:
from pynq import Overlay
from pynq import allocate
from pynq import MMIO
import numpy as np
import matplotlib.pyplot as plt
import time
import struct
import pickle
import socket

# Define relevant addreses
These addresses are extracted from the block design and from "xfft_wrapper_hw.h"

In [ ]:
FFT_WRAPPER_BASE_ADDRESS = 0x40000000
FFT_WRAPPER_ADDRESS_RANGE = 0x10000
XFFT_WRAPPER_AXI4L_IF_ADDR_AP_CTRL = 0x000
XFFT_WRAPPER_AXI4L_IF_ADDR_GIE = 0x004
XFFT_WRAPPER_AXI4L_IF_ADDR_IER = 0x008
XFFT_WRAPPER_AXI4L_IF_ADDR_ISR = 0x00c
XFFT_WRAPPER_AXI4L_IF_ADDR_WINDOW_COEFFS_BASE = 0x200
XFFT_WRAPPER_AXI4L_IF_ADDR_WINDOW_COEFFS_HIGH = 0x3ff

# Write bitstream into the FPGA

In [ ]:
overlay = Overlay('/home/xilinx/pynq/overlays/fft_project/fft_project_wrapper.bit')

# Enable microphone as input
And bypass audio during 5 seconds

In [ ]:
pAudio = overlay.audio_codec_ctrl_0
pAudio.configure()
pAudio.select_line_in()
pAudio.set_volume(40)
pAudio.bypass(seconds = 5)

# Create relevant objects

In [ ]:
fft_mmio = MMIO(FFT_WRAPPER_BASE_ADDRESS,FFT_WRAPPER_ADDRESS_RANGE) # Provides access to the FFT address map
fft_dma = overlay.fft_hier.fft_dma # It controls the AXI DMA IP

# Define relevant constants

In [ ]:
N = 256 # Number of FFT points

# Define ap_fixed conversion function

In [ ]:
def float_to_ap_fixed(values, total_bits, int_bits, signed=True):
    """
    Converts floating-point values to the raw bit pattern of an HLS
    ap_fixed<total_bits, int_bits> (signed=True) or
    ap_ufixed<total_bits, int_bits> (signed=False) type.

    Returns an unsigned integer array (uint8/16/32/64, whichever is the
    smallest that fits total_bits) holding the raw total_bits-wide
    two's-complement pattern in the low-order bits, ready to write directly
    into an MMIO register or DMA buffer.
    """
    values = np.asarray(values, dtype=np.float64)
    frac_bits = total_bits - int_bits
    scale = 2.0 ** frac_bits

    if signed:
        min_val = -(2 ** (total_bits - 1))
        max_val = 2 ** (total_bits - 1) - 1
    else:
        min_val = 0
        max_val = 2 ** total_bits - 1

    scaled = np.clip(np.round(values * scale), min_val, max_val).astype(np.int64)

    # True modulo (not bitwise) wraps negative values into their correct
    # two's-complement pattern, e.g. -1 mod 2**18 == 0x3FFFF
    raw = np.mod(scaled, 1 << total_bits)

    if total_bits <= 8:
        return raw.astype(np.uint8)
    elif total_bits <= 16:
        return raw.astype(np.uint16)
    elif total_bits <= 32:
        return raw.astype(np.uint32)
    elif total_bits <= 64:
        return raw.astype(np.uint64)
    else:
        raise ValueError("total_bits > 64 not supported")

def ap_fixed_to_float(raw, total_bits, int_bits, signed=True):
    """
    Converts the raw bit pattern of an HLS ap_fixed<total_bits, int_bits>
    (signed=True) or ap_ufixed<total_bits, int_bits> (signed=False) type
    back to a floating-point value. Accepts `raw` as either an unsigned or
    already sign-extended numpy integer array/scalar.
    """
    raw = np.mod(np.asarray(raw).astype(np.int64), 1 << total_bits)
    frac_bits = total_bits - int_bits
    scale = 2.0 ** frac_bits

    if signed:
        sign_bit = 1 << (total_bits - 1)
        raw = np.where(raw >= sign_bit, raw - (1 << total_bits), raw)

    return raw.astype(np.float64) / scale

# Program window coefficients before starting the FFT
The window coefficients are written into the AXI4-Lite slave memory map starting at `XFFT_WRAPPER_AXI4L_IF_ADDR_WINDOW_COEFFS_BASE` (offset `0x200`). Each coefficient is represented as `ap_ufixed<18,0>` (18 fractional bits).

In [ ]:
def program_window_coefficients(window_coeffs):
    """
    Programs the window coefficients into the FFT HLS IP via MMIO.

    Parameters:
    window_coeffs (array-like): Window coefficients as floats in [0.0, 1.0].
                                Length should be N/2 (128) or N (256, first N/2 used).
    """
    coeffs = np.asarray(window_coeffs, dtype=np.float64)
    if len(coeffs) > N // 2:
        coeffs = coeffs[: N // 2]

    # ap_ufixed<18,0>: 18 fractional bits, integer range [0, 2**18 - 1]
    fixed_coeffs = float_to_ap_fixed(coeffs,18,0,False)

    for i, val in enumerate(fixed_coeffs):
        addr = XFFT_WRAPPER_AXI4L_IF_ADDR_WINDOW_COEFFS_BASE + (i * 4)
        fft_mmio.write(addr, int(val))

# Program default rectangular window coefficients (all 1.0s) before starting the FFT
window_coeffs = np.ones(N // 2)
program_window_coefficients(window_coeffs)

In [ ]:
for i in range(4):
    addr = XFFT_WRAPPER_AXI4L_IF_ADDR_WINDOW_COEFFS_BASE + i*4
    print(hex(addr), hex(fft_mmio.read(addr)))

# Start the FFT HLS IP block

In [ ]:
# Set ap_start & auto_restart fields to '1'
fft_mmio.write(XFFT_WRAPPER_AXI4L_IF_ADDR_AP_CTRL,0x91)

# Read control register (it should be 0x81)
control = fft_mmio.read(XFFT_WRAPPER_AXI4L_IF_ADDR_AP_CTRL)
print(hex(control))

# Define function that calculates the FFT using the FPGA

In [ ]:
def fft_radix2_hls(input_samples, window_coeffs=None):
    # Program window coefficients if provided
    if window_coeffs is not None:
        program_window_coefficients(window_coeffs)
    N = len(input_samples)

    # Allocate AXI DMA input and output buffers
    ## Input samples are 32 bits integers
    input_buffer = allocate(shape=(N,), dtype=np.uint32)
    ## Output samples are complex numbers with 32 bits for the real and imaginary parts
    output_buffer = allocate(shape=(2*N,), dtype=np.int32)

    # Load input samples into input buffer
    for i in range(N):
        input_buffer[i] = np.uint32(input_samples[i])
        #print(hex(input_buffer[i]))
    # Set the ap_continue field to '1'
    fft_mmio.write(XFFT_WRAPPER_AXI4L_IF_ADDR_AP_CTRL,0x91)

    # Send input samples to the AXI DMA input channel and read output
    fft_dma.sendchannel.transfer(input_buffer)
    fft_dma.recvchannel.transfer(output_buffer)
    fft_dma.sendchannel.wait()
    fft_dma.recvchannel.wait()

    # Close buffers
    input_buffer.close()
    output_buffer.close()

    # Get real and imaginary part of output samples
    fft_result_real = ap_fixed_to_float(output_buffer[::2],32,10,True)
    fft_result_imag = ap_fixed_to_float(output_buffer[1::2],32,10,True)

    # Create array for the complex values
    fft_result = np.zeros(N,dtype=np.complex64)
    # Since the integer part is represented using 24 bits out of the 32 bits we need to divide the integer values by 2**8
    for i in range(N):
        fft_result[i] = np.double(fft_result_real[i]) + 1j*(np.double(fft_result_imag[i]))
    return fft_result

# Create a tone to test the IP

In [ ]:
fs = 48e3 # Sampling frequency
f = 10e3 # Tone frequency
t = np.arange(N)/fs # Time array
sin_wave = np.array(np.sin(2 * np.pi * f * t) * 2**16, dtype=np.int32) # Tone
sin_wave = sin_wave / np.max(np.abs(sin_wave))

# Plot tone
plt.figure()
plt.plot(t,sin_wave)
plt.show()

# Calculate FFT using the FPGA and using Numpy for comparison

In [ ]:
input_samples = sin_wave
input_samples_fixed = float_to_ap_fixed(input_samples,32,2,True)
start = time.time()
fft_fpga = np.abs(np.fft.fftshift(fft_radix2_hls(input_samples_fixed)))
end = time.time()
time_fpga = end-start
start = time.time()
fft_numpy = np.abs(np.fft.fftshift(np.fft.fft(input_samples)))
end = time.time()
time_numpy = end-start
print(f"FPGA = {time_fpga}")
print(f"NUMPY = {time_numpy}")

In [ ]:
print(max(fft_numpy)/max(fft_fpga))

#  Plot results

In [ ]:
freq_axis = np.fft.fftshift(np.fft.fftfreq(N, d=1/fs)) # Create frequency axis
plt.figure(figsize=(10,6))
plt.plot(freq_axis/1e3,fft_fpga)
plt.plot(freq_axis/1e3,fft_numpy)
plt.title("Frequency spectrum")
plt.ylabel("Amplitude")
plt.xlabel("Frequency [kHz]")
plt.show()

# Test FFT with a Hann window
To test with a proper window, we extract the first `N // 2` (128) coefficients of a Hann window (`np.hanning(N)[:N//2]`). These are programmed into the FPGA IP before calculating the FFT. In hardware, index `N // 2` is set to 1.0 and indices `N // 2 + 1` to `N - 1` mirror the lower half (`window_coeffs[N - i]`).

In [ ]:
# Extract the first N/2 unique coefficients of a Hann window
window_coeffs_hann = np.hanning(N)[: N // 2]

# Calculate FFT on FPGA using the programmed Hann window coefficients
start = time.time()
fft_fpga_windowed = np.abs(np.fft.fftshift(fft_radix2_hls(input_samples_fixed, window_coeffs=window_coeffs_hann)))
end = time.time()
time_fpga_windowed = end - start

# Reconstruct full symmetric window applied by hardware for NumPy reference
window_full = np.zeros(N)
window_full[: N // 2] = window_coeffs_hann
window_full[N // 2] = 1.0
window_full[N // 2 + 1 :] = window_coeffs_hann[:0:-1]

# Compute reference windowed FFT using NumPy
input_samples_windowed = input_samples * window_full
start = time.time()
fft_numpy_windowed = np.abs(np.fft.fftshift(np.fft.fft(input_samples_windowed)))
end = time.time()
time_numpy_windowed = end - start

print(f"FPGA (Hann Window) time = {time_fpga_windowed}")
print(f"NUMPY (Hann Window) time = {time_numpy_windowed}")

In [ ]:
# Plot windowed frequency spectrum comparison
plt.figure(figsize=(10, 6))
plt.plot(freq_axis / 1e3, fft_fpga_windowed, label="FPGA (Hann Window)")
plt.plot(freq_axis / 1e3, fft_numpy_windowed, '--', label="NumPy (Hann Window)")
plt.title("Frequency Spectrum with Hann Window")
plt.ylabel("Amplitude")
plt.xlabel("Frequency [kHz]")
plt.legend()
plt.show()

# Define function to get samples from microphone

In [ ]:
def get_samples_from_mic(N):
    sampling_time = N*4/pAudio.sample_rate
    pAudio.record(sampling_time)
    input_samples = np.int32((pAudio.buffer << 8) >> 8)[::2]
    input_sample_ds = input_samples[::4] # Down sampling to reduce bandwidth
    return input_sample_ds

In [ ]:
time_array = np.arange(N)/pAudio.sample_rate
input_samples = get_samples_from_mic(N)

# Plot microphone signal

In [ ]:
time_array = np.arange(N)/pAudio.sample_rate
input_samples = get_samples_from_mic(N)
plt.figure(figsize=(10,6))
plt.plot(time_array*1e3,input_samples)
plt.title("Microphone signal")
plt.ylabel("Amplitude")
plt.xlabel("Time [ms]")
plt.show()

# Send samples using sockets

In [ ]:
# Create a socket
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# Define the receiver's address and port
qt_server_ip = '192.168.2.1'
receiver_address = qt_server_ip
receiver_port = 12345

# Connect to the receiver
s.connect((receiver_address, receiver_port))

sending_time = 60

# Keep sending data for one minute
start_time = time.time()
while time.time() - start_time < sending_time:
    # Calculate FFT using the samples from the mic
    array_to_send = fft_radix2_hls(get_samples_from_mic(N))

    # Serialize and send the array
    data = pickle.dumps(array_to_send)
    s.send(data)

    # Wait for some time before sending the next data
    time.sleep(0.05)  # Adjust as needed

# Close the connection
s.close()